# 03 - Temporal Alignment & Aggregation

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.config import load_config
from src.utils import setup_logging, ensure_dir
from src.temporal_alignment import (
    normalize_to_peak, generate_period_windows,
    aggregate_periods, compute_derived_meteo,
)

logger = setup_logging()
config = load_config()
output_dir = config['data']['output_dir']

# ============================================================
# SWITCH: Aggregation mode
# Change this to run different aggregation strategies
# Options: "biweekly", "monthly", "phenological"
# ============================================================
AGGREGATION_MODE = config['aggregation']['mode']
print(f"Aggregation mode: {AGGREGATION_MODE}")

Aggregation mode: peak_stages


## 1. Load Data

In [2]:
spectral_df = pd.read_parquet(f'{output_dir}/spectral_raw.parquet')
spectral_df['date'] = pd.to_datetime(spectral_df['date'])

meteo_df = pd.read_parquet(f'{output_dir}/meteo_raw.parquet')
meteo_df['date'] = pd.to_datetime(meteo_df['date'])

static_df = pd.read_parquet(f'{output_dir}/static_features.parquet')
peak_df = pd.read_csv(f'{output_dir}/peak_days.csv')
peak_df['peak_date'] = pd.to_datetime(peak_df['peak_date'])

print(f"Spectral: {spectral_df.shape}")
print(f"Meteo: {meteo_df.shape}")
print(f"Static: {static_df.shape}")
print(f"Peak days: {len(peak_df)} fields")

Spectral: (27600, 44)
Meteo: (87600, 9)
Static: (240, 66)
Peak days: 240 fields


## 2. Apply Peak QC Filter

In [3]:
# SWITCH: Exclude anomalous peak fields
EXCLUDE_ANOMALOUS = config['peak_qc']['exclude_anomalous']

if EXCLUDE_ANOMALOUS:
    valid_peaks = peak_df[~peak_df['peak_anomalous']]
    print(f"Excluding {peak_df['peak_anomalous'].sum()} anomalous fields")
else:
    valid_peaks = peak_df.copy()
    print("Keeping all fields (anomalous flagged but not excluded)")

print(f"Fields for alignment: {len(valid_peaks)}")

Excluding 12 anomalous fields
Fields for alignment: 228


## 3. Normalize Time to Peak

In [4]:
spectral_norm = normalize_to_peak(spectral_df, valid_peaks)
meteo_norm = normalize_to_peak(meteo_df, valid_peaks)

# Compute derived meteo variables
gdd_base = config['meteo'].get('gdd_base_temp', 0.0)
meteo_norm = compute_derived_meteo(meteo_norm, gdd_base_temp=gdd_base)

print(f"Spectral normalized: {spectral_norm.shape}")
print(f"Meteo normalized: {meteo_norm.shape}")
print(f"Normalized day range: [{spectral_norm['normalized_day'].min()}, {spectral_norm['normalized_day'].max()}]")

13:51:33 | INFO    | Dropped 1380 rows (fields without peak date)


13:51:33 | INFO    | Time normalization: 228 fields, normalized_day range [-327, 88]
13:51:33 | INFO    | Dropped 4380 rows (fields without peak date)
13:51:33 | INFO    | Time normalization: 228 fields, normalized_day range [-329, 90]


Spectral normalized: (26220, 45)
Meteo normalized: (83220, 13)
Normalized day range: [-327, 88]


## 4. Define Period Windows

In [5]:
# Override config mode for this run
config['aggregation']['mode'] = AGGREGATION_MODE
windows = generate_period_windows(config)

if AGGREGATION_MODE in ("custom", "calendar_monthly", "calendar_biweekly"):
    print(f"\n{'Period':<25} {'Start Date':>12} {'End Date':>12} {'Days':>6}")
    print("-" * 60)
    for name, start, end in windows:
        duration = (end - start).days + 1
        print(f"{name:<25} {start.strftime('%Y-%m-%d'):>12} {end.strftime('%Y-%m-%d'):>12} {duration:>6}")
else:
    print(f"\n{'Period':<25} {'Start Day':>10} {'End Day':>10} {'Duration':>10}")
    print("-" * 60)
    for name, start, end in windows:
        print(f"{name:<25} {start:>10} {end:>10} {end - start + 1:>10}")

print(f"\nTotal periods: {len(windows)}")


Period                     Start Day    End Day   Duration
------------------------------------------------------------
cp1                             -300       -121        180
cp2                             -120        -60         61
cp3                              -59        -14         46
cp4                              -15         15         31
cp5                               16         36         21

Total periods: 5


## 5. Aggregate Features

In [6]:
# Determine feature columns
vi_cols = [c for c in config['spectral']['precomputed_indices'] if c in spectral_norm.columns]
band_cols = [c for c in config['spectral']['bands'] if c in spectral_norm.columns]
meteo_cols = [c for c in meteo_norm.columns if c not in ['field_key', 'date', 'normalized_day']]

print(f"Spectral features: {len(vi_cols)} VIs + {len(band_cols)} bands")
print(f"Meteo features: {len(meteo_cols)}")

features_agg = aggregate_periods(
    spectral_norm, meteo_norm, config,
    feature_columns=vi_cols + band_cols,
    meteo_columns=meteo_cols,
)

print(f"\nAggregated features: {features_agg.shape}")
features_agg.head()

13:51:36 | INFO    | Aggregating 42 spectral + 10 meteo features across 5 periods


Spectral features: 32 VIs + 10 bands
Meteo features: 10


13:52:13 | INFO    | Aggregation complete: 228 fields x 740 columns



Aggregated features: (228, 741)


,field_key,cp1_NDVI_mean,cp1_NDVI_std,cp1_NDVI_p10,cp1_NDVI_p90,cp1_EVI2_mean,cp1_EVI2_std,cp1_EVI2_p10,cp1_EVI2_p90,cp1_GNDVI_mean,...,cp5_t2m_mean_mean,cp5_t2m_min_mean,cp5_t2m_max_mean,cp5_precip_sum,cp5_pet_sum,cp5_dewpoint_mean,cp5_ssrd_MJm2_mean,cp5_vpd_mean,cp5_gdd_sum,cp5_water_deficit_mean
0,1,0.312688,0.225930,0.168731,0.741511,0.213141,0.153186,0.116731,0.457217,0.412578,...,18.227302,12.580666,24.224247,105.261258,145.764418,9.558025,21.972352,0.887993,386.451585,1.928722
1,10,0.335348,0.247005,0.148808,0.760488,0.224530,0.179964,0.098304,0.545348,0.416062,...,18.287523,12.668893,24.469796,126.503636,172.404543,9.079537,21.939255,0.930850,389.956239,2.185757
2,11,0.290572,0.213093,0.152718,0.719351,0.186674,0.142346,0.094367,0.469065,0.384489,...,17.435737,11.369960,24.114848,33.808981,189.322412,7.325013,25.215132,0.977351,372.590479,7.405401
3,12_,0.369190,0.273686,0.162399,0.832769,0.246069,0.206170,0.099199,0.609655,0.437823,...,18.269980,12.482185,24.498798,107.406012,173.153907,8.952804,22.097171,0.940850,388.300317,3.130852
4,14,0.340006,0.197640,0.196200,0.684861,0.209301,0.140798,0.118941,0.450615,0.414109,...,18.259302,12.800180,24.163561,98.507949,178.407848,9.694681,21.929602,0.884937,388.119279,3.804757


## 6. Merge with Static Features

In [7]:
# Merge with static features (soil, elevation, management)
merged = features_agg.merge(static_df, on='field_key', how='inner')
print(f"After merging static: {merged.shape}")
print(f"Fields with all data: {len(merged)}")

# Summary
n_temporal = len([c for c in merged.columns if any(c.startswith(w[0]) for w in windows)])
n_static = len(merged.columns) - n_temporal - 1  # -1 for field_key
print(f"Temporal features: ~{n_temporal}")
print(f"Static features: ~{n_static}")

After merging static: (228, 806)
Fields with all data: 228
Temporal features: ~740
Static features: ~65


## 7. Save Results

In [ ]:
output_path = f'{output_dir}/features_{AGGREGATION_MODE}.parquet'
merged.to_parquet(output_path, index=False)
print(f"Saved: {output_path}")
print(f"Shape: {merged.shape}")
print(f"\nmode: {AGGREGATION_MODE}")

Saved: data/processed/features_peak_stages.parquet
Shape: (228, 806)

Notebook 03 complete! (mode: peak_stages)
